In [ ]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False
import sys

print(sys.executable)  #
from pprint import pprint
import polars as pl
from pathlib import Path

In [ ]:
from groq import Groq

In [ ]:
from sec_10k_rag.common.common import write_to_parquet_file, load_parquet

In [ ]:
from sec_10k_rag.step_07_search_quality.config import get_config, ConfigType, get_llm_api_key, ROOT_DIR

In [ ]:
from sec_10k_rag.step_07_search_quality.lib import generate_questions, load_chunks_text_from_db, sample_chunks, get_question_vectors, find_nearest_neighbors

In [ ]:
print(dir(ConfigType)) 

In [ ]:
config = get_config(ConfigType.GENERATE_QUESTIONS)

In [ ]:
print(config)

In [ ]:
instructions = config.instructions

In [ ]:
chunks_df = load_chunks_text_from_db(instructions.db, instructions.chunks_table)

In [ ]:
display(chunks_df.head(10))

In [ ]:
sample_df = sample_chunks(chunks_df, n=instructions.sample_size, seed=instructions.seed)

In [ ]:
print(sample_df.height)

In [ ]:
print(chunks_df.height)

In [ ]:
pprint(sorted(m.id for m in Groq(api_key=get_llm_api_key()).models.list().data))

In [ ]:
questions = generate_questions(
    sample=sample_df,
    llm_info=instructions.llm_info,
    company_names=instructions.company_names
)

In [ ]:
display(questions)

In [ ]:
questions_path = (
    Path(config.processed_data_dir) / instructions.output_file_name
)

In [ ]:
print(questions_path)

In [ ]:
# write_to_parquet_file(questions, questions_path)

In [ ]:
loaded_questions_df = load_parquet(questions_path)

In [ ]:
print(loaded_questions_df.height)

In [ ]:
questions_vector_df = get_question_vectors(questions_path)

In [ ]:
display(questions_vector_df.head(10))

In [ ]:
pprint(config)

In [ ]:
neighbors_df = find_nearest_neighbors(questions_vector_df, instructions.db, instructions.chunks_table, top_k=5)

In [ ]:
with pl.Config(tbl_rows=-1): display(neighbors_df)

In [ ]:
from sec_10k_rag.step_07_search_quality.lib import find_keyword_matches, fuse_ranks, score, rank_of_source

In [ ]:
matches = find_keyword_matches(
    questions_vector_df, db=instructions.db, table="chunks", top_k=30,
    filter_columns=("ticker", "fiscal_year"),
)
matches.group_by("question_id").len().describe()

In [ ]:
vectors = find_nearest_neighbors(
    questions_vector_df, db=instructions.db, table="chunks", top_k=30,
    filter_columns=("ticker", "fiscal_year"),
)


In [ ]:
fused = fuse_ranks(vectors, matches, rrf_k=60, top_k=10)

In [ ]:
print(fused.height) 

In [ ]:
load_chunks_text_from_db(instructions.db, instructions.chunks_table).schema["chunk_index"]
# Int64

In [ ]:
print(score(rank_of_source(questions_vector_df, fused), (1, 3, 5, 10)))

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
model = CrossEncoder("BAAI/bge-reranker-base")

In [ ]:
print(type(model.model).__name__) 

In [ ]:
print(model.device)  

In [ ]:
model.predict([("What was Tesla's revenue?", "Total revenues were $97.7B.")])
# array([0.9...], dtype=float32)

In [1]:
from sec_10k_rag.step_08_mcp_server.lib.search_filings import search_filings

In [ ]:
search_filings("TSLA", 2024, "How many vehicles did Tesla deliver?")

In [2]:
search_filings("TSLA", 2024, "How many vehicles did Tesla deliver?", top_k=5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

rank,section_id,chunk_index,item,title,source_url,rerank_score,text
u32,str,i64,str,str,str,f32,str
1,"""TSLA-2024-item8""",12,"""8""","""Financial Statements and Suppl…","""https://www.sec.gov/Archives/e…",0.516656,"""Automotive leasing | 1,003 | 1…"
2,"""TSLA-2024-item1A""",7,"""1A""","""Risk Factors""","""https://www.sec.gov/Archives/e…",0.246783,"""Because we do not have indepen…"
3,"""TSLA-2024-item7""",1,"""7""","""Management's Discussion and An…","""https://www.sec.gov/Archives/e…",0.167944,"""In 2024, we produced approxima…"
4,"""TSLA-2024-item8""",11,"""8""","""Financial Statements and Suppl…","""https://www.sec.gov/Archives/e…",0.031067,"""3 | 3 Additional paid-in capit…"
5,"""TSLA-2024-item1A""",13,"""1A""","""Risk Factors""","""https://www.sec.gov/Archives/e…",0.025838,"""We face strong competition for…"


In [ ]:
from sec_10k_rag.common.common import start_db

In [ ]:
start_db()